# Topstep 50K Fast Pipeline

Run this notebook top-to-bottom. Set `run_mode` to `FAST` for quick iteration or `FULL` for full training.


In [1]:
# Setup
import os
import sys
import json as pyjson
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display, JSON

os.environ.setdefault("RISK_PRESET_NAME", "TOPSTEP_50K")
os.environ.setdefault("ES_BARS_H5", "data/processed/es_bars_2010_2025.h5")

run_mode = "FAST"  # FAST or FULL
data_path = os.environ["ES_BARS_H5"]
dataset_key = "bars_5min"
model_dir = "models/nn_saved"
fold = 0

fast_max_bars = 250_000

project_root = Path().resolve().parent if Path().resolve().name == 'analysis' else Path().resolve()
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from core.risk_presets import get_risk_preset
from core.selection import bars_per_day
from data.clean_bars import clean_bars
from features.labels_aligned import make_aligned_fixed_horizon_labels
from models.nn_inference import (
    load_nn_bundle,
    predict_scores_for_bars,
    artifact_compatibility_issues,
)
from backtesting.backtest import run_backtest_nn
from analysis.monte_carlo_combine import simulate_combine


In [2]:
# Load and clean bars
if not Path(data_path).exists():
    raise FileNotFoundError(f"Missing data file: {data_path}")

with pd.HDFStore(data_path, "r") as store:
    if dataset_key not in store:
        raise KeyError(f"Dataset key {dataset_key!r} not found in H5.")
    bars = store[dataset_key].copy()

bars["timestamp"] = pd.to_datetime(bars["timestamp"], utc=True)
bars = bars.sort_values("timestamp").reset_index(drop=True)

preset = get_risk_preset("TOPSTEP_50K")
bars = clean_bars(bars, tick_size=preset.risk_config.tick_size, verbose=True)
if run_mode.upper() == "FAST":
    bars = bars.tail(fast_max_bars).reset_index(drop=True)
assert bars["timestamp"].is_monotonic_increasing, "Bars must be time-ordered"

print(f"Loaded {len(bars):,} bars")
print(f"Date range: {bars['timestamp'].min()} to {bars['timestamp'].max()}")



BAR CLEANING REPORT
Input rows:  306,933
Output rows: 306,933  (removed 0)

Loaded 250,000 bars
Date range: 2013-04-26 16:35:00+00:00 to 2025-12-19 20:55:00+00:00


In [3]:
# Train or reuse NN artifacts
config_path = Path(model_dir) / f"fold_{fold}" / "config.json"

def needs_retrain(path: Path) -> bool:
    if not path.exists():
        return True
    try:
        cfg = pyjson.loads(path.read_text())
    except Exception:
        return True
    issues = artifact_compatibility_issues(cfg, strict_versions=True)
    if issues:
        print(f"Artifact incompatibility: {issues}")
        return True
    return False

if needs_retrain(config_path):
    train_cmd = [
        sys.executable,
        "models/nn_train.py",
        "--data-path",
        data_path,
        "--dataset-key",
        dataset_key,
        "--output-dir",
        model_dir,
    ]
    if run_mode.upper() == "FAST":
        train_cmd += ["--fast", "--max-bars", str(fast_max_bars)]
    print('Training command:', ' '.join(train_cmd))
    subprocess.run(train_cmd, check=True)
else:
    print(f"Using existing model artifacts at {config_path}")


Using existing model artifacts at models/nn_saved/fold_0/config.json


In [4]:
# Load model bundle + label diagnostics
bundle = load_nn_bundle(model_dir, fold=fold)
nn_cfg = bundle.config.get("nn_config", {})

nn_cfg_display = {k: (v.isoformat() if hasattr(v, "isoformat") else v) for k, v in nn_cfg.items()}
print("NN artifact config:")
display(JSON(nn_cfg_display, indent=2))

labels_df = make_aligned_fixed_horizon_labels(
    bars,
    horizon_bars=int(nn_cfg["horizon_bars"]),
    threshold_ticks=int(nn_cfg["threshold_ticks"]),
    tick_size=float(nn_cfg["tick_size"]),
    entry_price_col=str(nn_cfg.get("label_entry_price_col", "open")),
    exit_price_col=str(nn_cfg.get("label_exit_price_col", "close")),
)
print(f"Avg |ret_ticks|: {labels_df['ret_ticks'].abs().mean():.2f}")

prob_df = predict_scores_for_bars(bars, bundle)
scores = prob_df["score"].dropna()
if not scores.empty:
    percentiles = {f"p{p}": float(np.nanpercentile(scores, p)) for p in [50, 90, 95, 97, 98, 99, 99.5]}
else:
    percentiles = {}
print("Score percentiles:")
print(pyjson.dumps(percentiles, indent=2))


NN artifact config:


<IPython.core.display.JSON object>

Avg |ret_ticks|: 33.28
Score percentiles:
{
  "p50": 0.40742279048655605,
  "p90": 0.45705218551960025,
  "p95": 0.474061131074805,
  "p97": 0.4889555820103318,
  "p98": 0.5024061351693494,
  "p99": 0.5310871384890243,
  "p99.5": 0.5711316966752913
}


In [5]:
# Backtest (ML-only, aligned time-exit)
risk_cfg = preset.risk_config
results = run_backtest_nn(
    bars,
    prob_df,
    score_threshold=float(nn_cfg["score_threshold"]),
    selection_mode=str(nn_cfg.get("selection_mode", "global_threshold")),
    day_percentile_floor=float(nn_cfg.get("day_percentile_floor", 0.90)),
    global_floor_score=float(nn_cfg.get("global_floor_score", nn_cfg["score_threshold"])),
    max_trades_per_day=int(nn_cfg["max_trades_per_day"]),
    min_bars_between_trades=int(nn_cfg["min_bars_between_trades"]),
    enable_long=bool(nn_cfg["enable_long"]),
    enable_short=bool(nn_cfg["enable_short"]),
    horizon_bars=int(nn_cfg["horizon_bars"]),
    execution_mode=str(nn_cfg["execution_mode"]),
    exit_price_mode=str(nn_cfg["exit_price_mode"]),
    session_mode=str(nn_cfg["session_mode"]),
    deadline_time=nn_cfg.get("deadline_time"),
    deadline_relax_factor=float(nn_cfg.get("deadline_relax_factor", 0.98)),
    bar_minutes=int(nn_cfg["bar_minutes"]),
    session_start=risk_cfg.session_start,
    session_end=risk_cfg.session_end,
    stop_loss_ticks=int(nn_cfg["stop_loss_ticks"]),
    target_multiplier=float(nn_cfg["target_multiplier"]),
    catastrophic_stop_ticks=int(nn_cfg.get("catastrophic_stop_ticks", int(nn_cfg["threshold_ticks"]) * 4)),
    max_hold_bars=int(nn_cfg["max_hold_bars"]),
    tick_size=float(nn_cfg["tick_size"]),
    tick_value=float(nn_cfg["tick_value"]),
    save_trades_path="analysis/notebook_backtest_trades_50k.csv",
)

print("Backtest summary:")
print(pyjson.dumps(results["summary"], indent=2))
print("Daily stats:")
print(pyjson.dumps(results["daily_stats"], indent=2))
print("Exit reasons:")
print(pyjson.dumps(results.get("exit_reason_counts", {}), indent=2))
print("Exit reason avg PnL:")
print(pyjson.dumps(results.get("exit_reason_avg_pnl", {}), indent=2))

bars_day = bars_per_day(
    session_mode=str(nn_cfg.get("session_mode", "RTH")),
    session_start=risk_cfg.session_start,
    session_end=risk_cfg.session_end,
    bar_minutes=int(nn_cfg["bar_minutes"]),
)
print(f"Bars/day (session): {bars_day} | Target trades/day: {nn_cfg.get('target_trades_per_day')}")

trades = pd.read_csv("analysis/notebook_backtest_trades_50k.csv")
assert len(trades) > 0, "Backtest produced 0 trades; aborting."



Backtest summary:
{
  "trades": 489,
  "wins": 182,
  "losses": 307,
  "win_rate": 0.3721881390593047,
  "profit_factor": 0.6603806942070837,
  "gross_wins": 3922.1000000000004,
  "gross_losses": 5939.149999999999,
  "net_pnl": -2017.0499999999997,
  "avg_pnl": -4.124846625766871,
  "max_drawdown": 2017.0499999985768,
  "ending_equity": 47982.95000000142,
  "starting_balance": 50000.0
}
Daily stats:
{
  "total_trading_days": 3254,
  "avg_trades_per_day": 0.15027658266748617,
  "avg_trades_per_active_day": 1.1138952164009113,
  "max_trades_in_day": 2,
  "days_with_trades": 439,
  "days_with_zero_trades": 2815,
  "trades_per_day_distribution": {
    "0": 2815,
    "1": 389,
    "2": 50
  },
  "pct_days_with_1_trade": 11.954517516902275,
  "pct_days_with_2_trades": 1.5365703749231714
}
Exit reasons:
{
  "SESSION_FLAT": 407,
  "TIME_EXIT": 62,
  "CATASTOP": 20
}
Exit reason avg PnL:
{
  "CATASTOP": -72.825,
  "SESSION_FLAT": -4.395945945945946,
  "TIME_EXIT": 19.816129032258065
}
Bars/day 

In [6]:
# Trades diagnostics
trades_df = pd.DataFrame(results.get("trades", []))
if trades_df.empty:
    print("No trades produced.")
else:
    trades_df["entry_time"] = pd.to_datetime(trades_df["entry_time"], utc=True)
    trades_df["day"] = trades_df["entry_time"].dt.date
    trades_per_day = trades_df.groupby("day")["pnl"].count()
    print(f"Avg trades/day: {trades_per_day.mean():.2f}")
    print(f"Win rate: {(trades_df['pnl'] > 0).mean():.2%}")
    print(f"Avg pnl/trade: {trades_df['pnl'].mean():.2f}")
    wins = trades_df[trades_df['pnl'] > 0]['pnl'].sum()
    losses = trades_df[trades_df['pnl'] <= 0]['pnl'].sum()
    pf = wins / abs(losses) if losses != 0 else float('inf')
    print(f"Profit factor: {pf:.2f}")


Avg trades/day: 1.11
Win rate: 37.22%
Avg pnl/trade: -4.12
Profit factor: 0.66


In [7]:
# Monte Carlo combine pass-rate
if trades_df.empty:
    raise ValueError("No trades available for Monte Carlo simulation.")
combine = simulate_combine(
    trades_df,
    starting_balance=preset.risk_config.starting_balance,
    profit_target=preset.profit_target,
    daily_loss_limit=preset.risk_config.max_daily_loss,
    trailing_drawdown=preset.risk_config.trailing_drawdown,
    runs=5000 if run_mode.upper() == "FAST" else 20000,
    seed=42,
    max_days=252,
    consistency_limit=preset.consistency_limit,
)

print("Topstep 50K pass-rate summary:")
print(pyjson.dumps(combine, indent=2))
if combine.get('pass_rate', 0) > 0 and combine.get('days_to_pass'):
    print(f"Median days to pass: {combine['days_to_pass'].get('p50')}")


Topstep 50K pass-rate summary:
{
  "runs": 5000,
  "pass_rate": 0.0,
  "days_to_pass": {},
  "fail_reasons": {
    "daily_loss": 0,
    "trailing_drawdown": 372,
    "consistency_limit": 0,
    "max_days": 4628
  },
  "max_drawdown": {
    "p05": 650.6374999996915,
    "p50": 1322.0499999993772,
    "p95": 2007.1499999992957,
    "mean": 1330.3673499993288
  }
}
